# LayaLink

https://huggingface.co/sensein/layalink

In [1]:
!pip install "laya @ git+https://github.com/NandhaKishorM/laya.git@80a1da1fa5448ea12163f4d977935625a0aea670"


  Cloning https://github.com/NandhaKishorM/laya.git (to revision 80a1da1fa5448ea12163f4d977935625a0aea670) to /tmp/pip-install-cwqfp8p7/laya_738b0f99d99c4036a451bf49edd75ef4
  Running command git clone --filter=blob:none --quiet https://github.com/NandhaKishorM/laya.git /tmp/pip-install-cwqfp8p7/laya_738b0f99d99c4036a451bf49edd75ef4
  Running command git rev-parse -q --verify '80a1da1fa5448ea12163f4d977935625a0aea670^{commit}'
  80a1da1fa5448ea12163f4d977935625a0aea670
  Resolved https://github.com/NandhaKishorM/laya.git to commit 80a1da1fa5448ea12163f4d977935625a0aea670
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for laya: filename=laya-0.4.34-py3-none-any.whl size=505536 sha256=ed56e21389c87bf44ba14e219878063d539a303c157a101f133074684fb04623
  Stored in directory: /root/.cache/pip/wheels/1a/76/f9/1e0267bf8447a17e5db69b43e39669ff30d8f13576677ca31e
Successfully built laya
   ━━━━━

In [2]:
import laya

agent = laya.Agent("sensein/layalink", device="cuda")      # or "mps" (Apple Silicon) / "cpu"
# a local copy works the same way: laya.Agent("path/to/LayaLink")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
state = {'source': {'label': 'urinary bladder urothelium',
                    'definition': '',
                    'synonyms': ['transitional epithelium'],
                    'ontology': 'Adult Mouse Anatomy'},
         'target': {'label': 'Bladder Transitional Cell Epithelium',
                    'definition': '',
                    'synonyms': ['Bladder Urothelium', 'Bladder Transitional Epithelium'],
                    'parents': ['Urothelium'],
                    'ontology': 'NCI Thesaurus (anatomy)'}}

questions = {'relation': {'type': 'choice',
                          'instructions': 'Classify SOURCE relative to TARGET using the supplied '
                                          'definitions. Subclass direction matters. Association is not '
                                          'equivalence. Missing information means insufficient.',
                          'criteria': {'equivalent': 'Same class meaning and scope.',
                                       'source_narrower': 'Source is a proper subclass of target.',
                                       'source_broader': 'Target is a proper subclass of source.',
                                       'related': 'Explicit association but neither equivalence nor '
                                                  'subclass.',
                                       'no_match': 'Definitions establish different concepts; no mapping '
                                                   'is supported.',
                                       'insufficient': 'Available evidence cannot determine the '
                                                       'relation.'}},
             'exact_match': {'type': 'noul',
                             'instructions': 'Do the supplied definitions establish that source and target '
                                             'have the same class meaning and scope? False includes '
                                             'insufficient evidence.'}}

result = agent.predict(state, questions)


In [4]:
result['answers']['relation']  # -> 'equivalent'  (gold: 'equivalent')  {'equivalent': 0.9993, 'source_broader': 0.0002, 'source_narrower': 0.0001, 'related': 0.0001, 'no_match': 0.0001, 'insufficient': 0.0001}

{'type': 'choice',
 'choice': 'equivalent',
 'probabilities': {'equivalent': 0.9993,
  'source_narrower': 0.0001,
  'source_broader': 0.0002,
  'related': 0.0001,
  'no_match': 0.0001,
  'insufficient': 0.0001},
 'confidence': 0.9964,
 'answer_confidence': 0.9993,
 'action': {'act_probability': 1.0}}

In [5]:
result['answers']['exact_match']  # -> 'true'  (gold: 'true')  {'true': 0.9997, 'false': 0.0003}

{'type': 'noul',
 'noul': 0.9997,
 'confidence': 0.9997,
 'answer_confidence': 0.9997,
 'action': {'act_probability': 1.0}}

## Using with Router

In [12]:
from laya import Router

router =  Router(models={"english": "sensein/layalink", "multilingual": "sensein/layalink"}, max_loaded=1, device="cuda")

In [13]:
state = {'source': {'label': 'urinary bladder urothelium',
                    'definition': '',
                    'synonyms': ['transitional epithelium'],
                    'ontology': 'Adult Mouse Anatomy'},
         'target': {'label': 'Bladder Transitional Cell Epithelium',
                    'definition': '',
                    'synonyms': ['Bladder Urothelium', 'Bladder Transitional Epithelium'],
                    'parents': ['Urothelium'],
                    'ontology': 'NCI Thesaurus (anatomy)'}}

questions = {'relation': {'type': 'choice',
                          'instructions': 'Classify SOURCE relative to TARGET using the supplied '
                                          'definitions. Subclass direction matters. Association is not '
                                          'equivalence. Missing information means insufficient.',
                          'criteria': {'equivalent': 'Same class meaning and scope.',
                                       'source_narrower': 'Source is a proper subclass of target.',
                                       'source_broader': 'Target is a proper subclass of source.',
                                       'related': 'Explicit association but neither equivalence nor '
                                                  'subclass.',
                                       'no_match': 'Definitions establish different concepts; no mapping '
                                                   'is supported.',
                                       'insufficient': 'Available evidence cannot determine the '
                                                       'relation.'}},
             'exact_match': {'type': 'noul',
                             'instructions': 'Do the supplied definitions establish that source and target '
                                             'have the same class meaning and scope? False includes '
                                             'insufficient evidence.'}}

In [14]:
result = router.predict(state, questions)
print(result["routing"]["repo"])          # sensein/layalink

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

sensein/layalink


In [18]:
import json
print(json.dumps(result, indent=2))

{
  "model": "laya-rl-agent",
  "answers": {
    "relation": {
      "type": "choice",
      "choice": "equivalent",
      "probabilities": {
        "equivalent": 0.9993,
        "source_narrower": 0.0001,
        "source_broader": 0.0002,
        "related": 0.0001,
        "no_match": 0.0001,
        "insufficient": 0.0001
      },
      "confidence": 0.9964,
      "answer_confidence": 0.9993,
      "action": {
        "act_probability": 1.0
      }
    },
    "exact_match": {
      "type": "noul",
      "noul": 0.9997,
      "confidence": 0.9997,
      "answer_confidence": 0.9997,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 371,
    "output_tokens": 0,
    "state_tokens": 106,
    "state_tokens_dropped": 0,
    "truncated": false,
    "truncated_questions": []
  },
  "routing": {
    "model": "multilingual",
    "repo": "sensein/layalink",
    "reason": "Latin script, language not identified and no non-English letters; using d